In [1]:
# Load in the data
import numpy as np
import pandas as pd
import kagglehub
import os
import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Downloads
nltk.download("stopwords")
nltk.download("punkt")
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

# Load spaCy
nlp = spacy.load("en_core_web_sm")

# Stopwords
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
root = kagglehub.dataset_download("melissamonfared/sephora-skincare-reviews")

# List files in the downloaded directory to verify file names
downloaded_files = os.listdir(root)
print("Files in downloaded directory:", downloaded_files)

file_path_1 = os.path.join(root, "product_info.csv")
file_path_2 = os.path.join(root, "reviews_0-250_masked.csv")
file_path_3 = os.path.join(root, "reviews_250-500_masked.csv")
file_path_4 = os.path.join(root, "reviews_500-750_masked.csv")
file_path_5 = os.path.join(root, "reviews_750-1250_masked.csv")
file_path_6 = os.path.join(root, "reviews_1250-end_masked.csv")

prod_info_df = pd.read_csv(file_path_1)
review_df_1 = pd.read_csv(file_path_2)
review_df_2 = pd.read_csv(file_path_3)
review_df_3 = pd.read_csv(file_path_4)
review_df_4 = pd.read_csv(file_path_5)
review_df_5 = pd.read_csv(file_path_6)


print(prod_info_df.head())
print(review_df_1.head())
print(review_df_2.head())
print(review_df_3.head())
print(review_df_4.head())

# Concatenate the review dataframes
concatenated_reviews_df = pd.concat([review_df_1, review_df_2, review_df_3, review_df_4, review_df_5], ignore_index=True)

# Merge the concatenated reviews dataframe with the product information dataframe
merged_reviews_df = pd.merge(concatenated_reviews_df, prod_info_df, on="product_id")

display(merged_reviews_df)

Using Colab cache for faster access to the 'sephora-skincare-reviews' dataset.
Files in downloaded directory: ['reviews_750-1250_masked.csv', 'product_info.csv', 'reviews_1250-end_masked.csv', 'reviews_500-750_masked.csv', 'reviews_250-500_masked.csv', 'reviews_0-250_masked.csv', 'product_info_skincare.csv']
  product_id               product_name  brand_id brand_name  loves_count  \
0    P473671    Fragrance Discovery Set      6342      19-69         6320   
1    P473668    La Habana Eau de Parfum      6342      19-69         3827   
2    P473662  Rainbow Bar Eau de Parfum      6342      19-69         3253   
3    P473660       Kasbah Eau de Parfum      6342      19-69         3018   
4    P473658  Purple Haze Eau de Parfum      6342      19-69         2691   

   rating  reviews            size                      variation_type  \
0  3.6364     11.0             NaN                                 NaN   
1  4.1538     13.0  3.4 oz/ 100 mL  Size + Concentration + Formulation   
2  4.

,Unnamed: 0.1,Unnamed: 0,rating_x,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,0,5,1.0,1.000000,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,1,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,2,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,3,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,4,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285407,49929,49929,5,1.0,NaN,0,0,0,2023-02-25,I wanted to test this out for a little bit bef...,...,0,0,0,"['Vegan', 'Clean + Planet Positive', 'Good for...",Skincare,Treatments,Blemish & Acne Treatments,0,NaN,NaN
285408,49935,49935,5,1.0,NaN,0,0,0,2023-02-12,I love Tara Harper but started to only use oil...,...,1,0,0,"['Vegan', 'Good for: Dullness/Uneven Texture',...",Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN
285409,49938,49938,3,0.0,0.529412,17,8,9,2022-05-29,Im heading into my 30s so I wanted to splurge ...,...,1,0,0,"['Radiant Finish', 'Good for: Dullness/Uneven ...",Skincare,Masks,Face Masks,0,NaN,NaN
285410,49942,49942,5,1.0,1.000000,3,0,3,2023-02-25,Really good for dry patchy skin!!! My skin has...,...,0,0,0,['Best for Dry Skin'],Skincare,Cleansers,Face Wash & Cleansers,0,NaN,NaN


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
print(merged_reviews_df.isna().sum())

Unnamed: 0.1                     0
Unnamed: 0                       0
rating_x                         0
is_recommended               57232
helpfulness                 154322
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                    345
review_title                 79694
skin_tone                    55226
eye_color                    67746
skin_type                    36874
hair_color                   71081
product_id                       0
product_name_x                   0
brand_name_x                     0
price_usd_x                      0
product_name_y                   0
brand_id                         0
brand_name_y                     0
loves_count                      0
rating_y                         0
reviews                          0
size                          6097
variation_type                8721
variation_value              10728
variation_desc      

In [5]:
merged_reviews_df['skin_tone'].unique()

array([nan, 'light', 'lightMedium', 'fairLight', 'fair', 'medium',
       'notSureST', 'mediumTan', 'tan', 'rich', 'olive', 'deep',
       'porcelain', 'dark', 'ebony'], dtype=object)

In [6]:
merged_reviews_df.dtypes
merged_reviews_df.dropna(subset=['review_text', 'review_title'], inplace=True)

In [7]:
# df type of review_text
print(merged_reviews_df["review_text"].dtype)

object


In [8]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove special chars, numbers
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# Apply preprocessing
merged_reviews_df["clean_review_text"] = merged_reviews_df["review_text"].apply(preprocess_text)
merged_reviews_df["clean_review_title"] = merged_reviews_df["review_title"].apply(preprocess_text)
print(merged_reviews_df)

        Unnamed: 0.1  Unnamed: 0  rating_x  is_recommended  helpfulness  \
0                  0           0         5             1.0     1.000000   
1                  1           1         1             0.0          NaN   
2                  2           2         5             1.0          NaN   
3                  3           3         5             1.0          NaN   
4                  4           4         5             1.0          NaN   
...              ...         ...       ...             ...          ...   
285407         49929       49929         5             1.0          NaN   
285408         49935       49935         5             1.0          NaN   
285409         49938       49938         3             0.0     0.529412   
285410         49942       49942         5             1.0     1.000000   
285411         49943       49943         5             1.0     1.000000   

        total_feedback_count  total_neg_feedback_count  \
0                          2             

In [9]:
# Replace empty reviews with placeholder
merged_reviews_df["review_text"] = merged_reviews_df["review_text"].fillna("no_review").astype(str)
merged_reviews_df.loc[merged_reviews_df["review_text"].str.strip() == "", "review_text"] = "no_review"

# Drop duplicate reviews
merged_reviews_df = merged_reviews_df.drop_duplicates(subset=["review_text"])


In [10]:
'''Text Standardization: Convert text to lowercase, handle contractions
(e.g., "don't" → "do not"), and normalize whitespace
Feature Engineering: Extract review length, rating discrepancy
(difference between text sentiment and numerical rating), and temporal features from review dates'''
#convert text to lowercase
merged_reviews_df["review_text"] = merged_reviews_df["review_text"].str.lower()
merged_reviews_df["review_title"] = merged_reviews_df["review_title"].str.lower()

#contractions
merged_reviews_df["review_text"] = merged_reviews_df["review_text"].apply(lambda x: re.sub(r"won't", "will not", x))


/tmp/ipython-input-2005887165.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_reviews_df["review_text"] = merged_reviews_df["review_text"].str.lower()
/tmp/ipython-input-2005887165.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_reviews_df["review_title"] = merged_reviews_df["review_title"].str.lower()
/tmp/ipython-input-2005887165.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



In [11]:
# count null values in 'is_reccomended'
merged_reviews_df['is_recommended'].isnull().sum()

np.int64(44059)

## creating the neural network

In [14]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 818.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00


In [15]:
# more imports
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow.keras as keras
import time

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [16]:
# create labeled examples
y = merged_reviews_df['is_recommended'] # label
X = merged_reviews_df['review_text'] # feature

In [17]:
# create training & test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [18]:
# implement TF-IDF to transform features into numerical vectors

# create TfidfVectorizer object
tfidf_vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1,2))

# fit vectorizer
tfidf_vectorizer.fit(X_train)

# transform train data
X_train_tfidf = tfidf_vectorizer.transform(X_train)

# transform test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
# find dimensionality of each training ex
vocab_size = len(tfidf_vectorizer.vocabulary_)
print(vocab_size)

In [ ]:
# neural network structure
nn_model = keras.Sequential() # model object

# create each layer and add to model object
input_layer = keras.layers.InputLayer(input_shape=(vocab_size,), name='input')
nn_model.add(input_layer)

hidden_1 = keras.layers.Dense(units=64, activation='relu', name='hl_1')
nn_model.add(hidden_1)
nn_model.add(keras.layers.Dropout(.25))

hidden_2 = keras.layers.Dense(units=32, activation='relu', name='hl_2')
nn_model.add(hidden_2)

hidden_3 = keras.layers.Dense(units=16, activation='relu', name='hl_3')
nn_model.add(hidden_3)

output_layer = keras.layers.Dense(units=1, activation='sigmoid', name='output')
nn_model.add(output_layer)

# summary of our model structure
nn_model.summary()

In [ ]:
# define optimization and loss functions
sgd_optimizer = keras.optimizers.SGD(learning_rate=0.1)
loss = keras.losses.BinaryCrossentropy(from_logits=False)

In [ ]:
# compile model
nn_model.compile(optimizer=sgd_optimizer, loss=loss, metrics=['accuracy'])

# fit model on training data
class ProgBarLoggerNEpochs(keras.callbacks.Callback):

    def __init__(self, num_epochs: int, every_n: int = 50):
        self.num_epochs = num_epochs
        self.every_n = every_n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            s = 'Epoch [{}/ {}]'.format(epoch + 1, self.num_epochs)
            logs_s = ['{}: {:.4f}'.format(k.capitalize(), v)
                      for k, v in logs.items()]
            s_list = [s] + logs_s
            print(', '.join(s_list))

# start training time
start_time = time.time()

# num of epochs
num_epochs = 30

history = nn_model.fit(X_train_tfidf,
                       y_train,
                       epochs=num_epochs,
                       verbose=0,
                       validation_split=0.2,
                       callbacks=[ProgBarLoggerNEpochs(num_epochs, every_n=5)])

# stop training time
end_time = time.time()

# elapsed time
print('Elapsed time: %.2fs' % (end_time-start_time))

Testing the accuracy of the Sentiment Analysis Model

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report

In [ ]:
# Making predictions on the test set

probability_predictions = nn_model.predict(X_test_tfidf.toarray())

print("Predictions for the first 20 examples:")
for i in range(0,20,1):
    if (probability_predictions[i]>0.5):
        print(probability_predictions[i],y_test.to_numpy()[i])

In [ ]:
# Evaluating the performance of the model on the test set

loss, accuracy = nn_model.evaluate(X_test_tfidf.toarray(),y_test)
print('Loss: ', str(loss) , 'Accuracy: ', str(accuracy))

In [ ]:
# Generating a confusion matrix
y_pred_probs = nn_model.predict(X_test)  # gives probabilities
y_pred = (y_pred_probs > 0.5).astype("int32")  # applying threshold 0.5
labels = ['positive', 'negative','neutral']
cm = confusion_matrix(y_test, y_pred,labels=labels)
cm

In [ ]:
# Calculating precision, recall, and f1 score
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Using the Classification report function
print(classification_report(y_test, y_pred))